In [1]:
import sys
import os
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir)
from system.controllers import RobotControllers as rcontrollers
import system.robot as robotSystem

In [ ]:
alpha = robotSystem.RobotDynamics(use_jit=False)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
path_to_urdf = os.path.join(
    project_root,
    'usage',
    'urdf','reach_alpha_5',
    'alpha_5_robot.urdf'
)
alpha.from_file(path_to_urdf)

root = "base_link"
tip = "alpha_standard_jaws_base_link"

kinematic_dict, K, P, L, D, C, g, B, qdd, joint_torque, sys_id_coeff , F_payload = alpha.build_model(root, tip,
                                                                                                      floating_base=False, 
                                                                                                      has_endeffector=True)

In [ ]:
arm_controllers = rcontrollers(4, alpha)

arm_pid, _ = arm_controllers.build_arm_pid()
tracking_pid = arm_controllers.trajectorytracking_pid()
sliding_mode = arm_controllers.sliding_mode()

In [4]:
# u_sat, err, buf_next = tracking_pid(
#     arm_controllers.q, 
#     arm_controllers.q_dot, 
#     arm_controllers.q_ref, 
#     arm_controllers.dq_ref,
#     arm_controllers.dqq_ref,
#     arm_controllers.Kp, 
#     arm_controllers.Ki, 
#     arm_controllers.Kd, 
#     arm_controllers.sum_e, 
#     arm_controllers.dt, 
#     arm_controllers.u_max, 
#     arm_controllers.u_min,
#     arm_controllers.sim_params
# )


In [5]:
arm_pid.save("arm_pid.casadi")
tracking_pid.save("tracking_pid.casadi")
sliding_mode.save("arm_sliding_mode_controller.casadi")
# Native form of the same function; runtime consumers need no source imports.
import casadi as ca
import subprocess
generator = ca.CodeGenerator('arm_sliding_mode.c', {'with_header': True})
generator.add(sliding_mode)
generator.generate()
subprocess.run(['gcc', '-O1', '-fPIC', '-shared', 'arm_sliding_mode.c',
                '-o', 'libarm_sliding_mode_controller.so', '-lm'], check=True)
